## Day 16 — Project Expenses (Clean CSV + Weekly Summary)

1) Load Raw Data

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("..") / "data"
RAW_PATH = DATA_DIR / "expenses.csv"
RAW_PATH


WindowsPath('../data/expenses.csv')

2) Read CSV

In [2]:
df = pd.read_csv(RAW_PATH)
df.head()


,date,type,category,amount,payment_method,description
0,2026-01-01,income,Income,1200000,transfer,Monthly salary
1,2026-01-01,expense,Bills,85000,cash,Internet subscription
2,2026-01-02,expense,Food,18000,cash,Grocery items
3,2026-01-02,expense,Transport,5000,cash,Taxi ride
4,2026-01-03,expense,Shopping,25000,card,House supplies


3) Parse Date

In [3]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df[["date"]].head()


,date
0,2026-01-01
1,2026-01-01
2,2026-01-02
3,2026-01-02
4,2026-01-03


4) Convert Amount to Numeric

In [4]:
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df[["amount"]].head()


,amount
0,1200000
1,85000
2,18000
3,5000
4,25000


5) Handle Missing Values

In [5]:
df["category"] = df["category"].fillna("Unknown")
df["payment_method"] = df["payment_method"].fillna("Unknown")
df["type"] = df["type"].fillna("Unknown")
df["description"] = df["description"].fillna("")

# Drop rows that have missing critical fields
df = df.dropna(subset=["date", "amount"])

df.isna().sum()


date              0
type              0
category          0
amount            0
payment_method    0
description       0
dtype: int64

6) Derive Day Name, Week Index, and Weekend Flag

In [6]:
df = df.sort_values("date")

df["day_name"] = df["date"].dt.day_name()

start_date = df["date"].min()
df["week"] = (((df["date"] - start_date).dt.days) // 7 + 1).astype("int64")

df["is_weekend"] = df["date"].dt.weekday >= 5

df[["date", "day_name", "week", "is_weekend"]].head()


,date,day_name,week,is_weekend
0,2026-01-01,Thursday,1,False
1,2026-01-01,Thursday,1,False
2,2026-01-02,Friday,1,False
3,2026-01-02,Friday,1,False
4,2026-01-03,Saturday,1,True


7) Add Signed Amount

In [7]:
df["type"] = df["type"].str.lower().str.strip()

# income -> +amount, expense -> -amount
df["signed_amount"] = df["amount"].where(df["type"] == "income", -df["amount"])

df[["date", "type", "amount", "signed_amount"]].head()


,date,type,amount,signed_amount
0,2026-01-01,income,1200000,1200000
1,2026-01-01,expense,85000,-85000
2,2026-01-02,expense,18000,-18000
3,2026-01-02,expense,5000,-5000
4,2026-01-03,expense,25000,-25000


8) Weekly Net Summary (Optional)

In [8]:
weekly = df.groupby("week", as_index=False)["signed_amount"].sum()
weekly = weekly.rename(columns={"signed_amount": "net_total"})
weekly


,week,net_total
0,1,995000
1,2,-225000
2,3,-31000


9) Save Weekly Summary to Reports (Optional)

In [9]:
REPORTS_DIR = Path("..") / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

weekly_path = REPORTS_DIR / "weekly_summary.csv"
weekly.to_csv(weekly_path, index=False)

print("Saved:", weekly_path)


Saved: ..\reports\weekly_summary.csv


10) Save Clean Dataset

In [10]:
clean_path = DATA_DIR / "clean_expenses.csv"
df.to_csv(clean_path, index=False)

print("Saved:", clean_path)


Saved: ..\data\clean_expenses.csv


11) Final Quick Check

In [11]:
print("Rows:", len(df))
df.describe(include="all")


Rows: 30


,date,type,category,amount,payment_method,description,day_name,week,is_weekend,signed_amount
count,30,30,30,3.000000e+01,30,30,30,30.000000,30,3.000000e+01
unique,NaN,2,5,NaN,4,25,7,NaN,2,NaN
top,NaN,expense,Food,NaN,cash,Grocery items,Thursday,NaN,False,NaN
freq,NaN,27,9,NaN,20,3,6,NaN,22,NaN
mean,2026-01-08 00:00:00,NaN,NaN,6.370000e+04,NaN,NaN,NaN,1.600000,NaN,2.463333e+04
min,2026-01-01 00:00:00,NaN,NaN,4.000000e+03,NaN,NaN,NaN,1.000000,NaN,-8.500000e+04
25%,2026-01-04 06:00:00,NaN,NaN,1.125000e+04,NaN,NaN,NaN,1.000000,NaN,-2.275000e+04
50%,2026-01-08 00:00:00,NaN,NaN,1.800000e+04,NaN,NaN,NaN,2.000000,NaN,-1.500000e+04
75%,2026-01-11 18:00:00,NaN,NaN,2.900000e+04,NaN,NaN,NaN,2.000000,NaN,-8.250000e+03
max,2026-01-15 00:00:00,NaN,NaN,1.200000e+06,NaN,NaN,NaN,3.000000,NaN,1.200000e+06


## Day 17 — Dashboard 1: Summary Tables

1) Totals: Income vs Expense + Net Profit

In [12]:
import pandas as pd
from pathlib import Path

# Ensure type is clean
df["type"] = df["type"].str.lower().str.strip()

# Create signed_amount if not exists
if "signed_amount" not in df.columns:
    df["signed_amount"] = df["amount"].where(df["type"] == "income", -df["amount"])

income_total = df.loc[df["type"] == "income", "amount"].sum()
expense_total = df.loc[df["type"] == "expense", "amount"].sum()
net_profit = df["signed_amount"].sum()

totals = pd.DataFrame({
    "metric": ["income_total", "expense_total", "net_profit"],
    "value": [income_total, expense_total, net_profit]
})

totals


,metric,value
0,income_total,1325000
1,expense_total,586000
2,net_profit,739000


2) Expense by Category

In [13]:
expense_by_category = (
    df[df["type"] == "expense"]
    .groupby("category", as_index=False)["amount"]
    .sum()
    .rename(columns={"amount": "expense_total"})
    .sort_values("expense_total", ascending=False)
)

expense_by_category


,category,expense_total
0,Bills,210000
2,Shopping,171000
1,Food,154000
3,Transport,51000


3) Top 5 Expenses

In [14]:
top5_expenses = (
    df[df["type"] == "expense"]
    .sort_values("amount", ascending=False)
    .head(5)
    [["date", "category", "amount", "payment_method", "description", "week"]]
    .copy()
)

top5_expenses.insert(0, "rank", range(1, len(top5_expenses) + 1))

top5_expenses


,rank,date,category,amount,payment_method,description,week
1,1,2026-01-01,Bills,85000,cash,Internet subscription,1
20,2,2026-01-11,Shopping,65000,card,Small electronics,2
17,3,2026-01-09,Bills,50000,transfer,Electricity bill,2
9,4,2026-01-05,Shopping,40000,card,Clothes,1
7,5,2026-01-04,Bills,30000,transfer,Mobile recharge,1


In [ ]:
4) Weekly Table (Income / Expense / Net)

In [15]:
weekly_table = (
    df.groupby(["week", "type"])["amount"]
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

# Ensure columns exist even if a week has only income or only expense
if "income" not in weekly_table.columns:
    weekly_table["income"] = 0
if "expense" not in weekly_table.columns:
    weekly_table["expense"] = 0

weekly_table["net_total"] = weekly_table["income"] - weekly_table["expense"]

weekly_table = weekly_table[["week", "income", "expense", "net_total"]].sort_values("week")

weekly_table


type,week,income,expense,net_total
0,1,1275000,280000,995000
1,2,50000,275000,-225000
2,3,0,31000,-31000


5) Save ALL tables into ONE CSV: summary_tables.csv

In [16]:
REPORTS_DIR = Path("..") / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = REPORTS_DIR / "summary_tables.csv"

# Convert each table to a tagged format
totals_out = totals.copy()
totals_out.insert(0, "table", "totals")

cat_out = expense_by_category.copy()
cat_out.insert(0, "table", "expense_by_category")

top5_out = top5_expenses.copy()
top5_out.insert(0, "table", "top5_expenses")

weekly_out = weekly_table.copy()
weekly_out.insert(0, "table", "weekly_table")

# Combine into one file (different columns are allowed; missing becomes empty)
summary_tables = pd.concat([totals_out, cat_out, top5_out, weekly_out], ignore_index=True, sort=False)

summary_tables.to_csv(OUT_PATH, index=False, encoding="utf-8")
print("Saved:", OUT_PATH)

summary_tables.head(20)


Saved: ..\reports\summary_tables.csv


,table,metric,value,category,expense_total,rank,date,amount,payment_method,description,week,income,expense,net_total
0,totals,income_total,1325000.0,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,totals,expense_total,586000.0,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,totals,net_profit,739000.0,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,expense_by_category,NaN,NaN,Bills,210000.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,expense_by_category,NaN,NaN,Shopping,171000.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,expense_by_category,NaN,NaN,Food,154000.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,expense_by_category,NaN,NaN,Transport,51000.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,top5_expenses,NaN,NaN,Bills,NaN,1.0,2026-01-01,85000.0,cash,Internet subscription,1.0,NaN,NaN,NaN
8,top5_expenses,NaN,NaN,Shopping,NaN,2.0,2026-01-11,65000.0,card,Small electronics,2.0,NaN,NaN,NaN
9,top5_expenses,NaN,NaN,Bills,NaN,3.0,2026-01-09,50000.0,transfer,Electricity bill,2.0,NaN,NaN,NaN
